In [1]:

import random
from concurrent.futures import ThreadPoolExecutor

# ---------------------------------------
# 1. FITNESS FUNCTION
# ---------------------------------------

def calculate_fitness(grid, sunlight, penalty):
    rows = len(grid)
    cols = len(grid[0])

    energy = 0
    shadow_count = 0

    # Calculate total sunlight energy
    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1:
                energy += sunlight[i][j]

    # Check adjacent panels (8 directions).
    # Each adjacent pair is counted only once.
    directions = [
        (0, 1), (1, -1), (1, 0), (1, 1)
    ]

    for i in range(rows):
        for j in range(cols):
            if grid[i][j] == 1:
                for dr, dc in directions:
                    ni = i + dr
                    nj = j + dc

                    if (0 <= ni < rows and
                            0 <= nj < cols):
                        if grid[ni][nj] == 1:
                            shadow_count += 1

    fitness = energy - (penalty * shadow_count)

    return fitness


# ---------------------------------------
# 2. INITIAL POPULATION
# ---------------------------------------

def create_population(rows, cols, k, population_size):
    population = []

    for _ in range(population_size):

        grid = [[0 for _ in range(cols)]
                for _ in range(rows)]

        total_cells = rows * cols

        # Randomly select number of panels
        panel_count = random.randint(1, min(k, total_cells))

        positions = random.sample(
            range(total_cells), panel_count
        )

        for position in positions:
            i = position // cols
            j = position % cols
            grid[i][j] = 1

        population.append(grid)

    return population


# ---------------------------------------
# 3. PARALLEL FITNESS EVALUATION
# ---------------------------------------

def parallel_fitness(population, sunlight, penalty):

    with ThreadPoolExecutor() as executor:

        fitness_values = list(
            executor.map(
                calculate_fitness,
                population,
                [sunlight] * len(population),
                [penalty] * len(population)
            )
        )

    return fitness_values


# ---------------------------------------
# 4. CREATE NEW CONFIGURATION
# ---------------------------------------

def create_new_configuration(current, best, k):

    rows = len(current)
    cols = len(current[0])

    new_grid = []

    # Combine cells from current and best configurations
    for i in range(rows):
        row = []

        for j in range(cols):
            if random.random() < 0.5:
                row.append(current[i][j])
            else:
                row.append(best[i][j])

        new_grid.append(row)

    # Enforce maximum number of panels
    positions = [
        (i, j)
        for i in range(rows)
        for j in range(cols)
        if new_grid[i][j] == 1
    ]

    random.shuffle(positions)

    for i, j in positions[k:]:
        new_grid[i][j] = 0

    # Randomly shift one panel to an empty neighbouring cell
    positions = [
        (i, j)
        for i in range(rows)
        for j in range(cols)
        if new_grid[i][j] == 1
    ]

    if positions:
        i, j = random.choice(positions)

        neighbours = []

        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:

                if dr == 0 and dc == 0:
                    continue

                ni = i + dr
                nj = j + dc

                if (0 <= ni < rows and
                        0 <= nj < cols and
                        new_grid[ni][nj] == 0):

                    neighbours.append((ni, nj))

        # If no empty neighbour exists, use any empty cell
        if not neighbours:
            neighbours = [
                (x, y)
                for x in range(rows)
                for y in range(cols)
                if new_grid[x][y] == 0
            ]

        if neighbours:
            ni, nj = random.choice(neighbours)

            new_grid[i][j] = 0
            new_grid[ni][nj] = 1

    return new_grid


# ---------------------------------------
# 5. PARALLEL CELLULAR ALGORITHM
# ---------------------------------------

def solar_panel_optimization(
        sunlight, k, penalty,
        population_size, iterations):

    rows = len(sunlight)
    cols = len(sunlight[0])

    # Step 1: Initialize population
    population = create_population(
        rows, cols, k, population_size
    )

    # Evaluate initial population
    fitness_values = parallel_fitness(
        population, sunlight, penalty
    )

    best_index = fitness_values.index(max(fitness_values))

    global_best = [
        row[:] for row in population[best_index]
    ]

    global_best_fitness = fitness_values[best_index]

    # Step 2: Repeat for given iterations
    for iteration in range(iterations):

        # Parallel fitness evaluation
        fitness_values = parallel_fitness(
            population, sunlight, penalty
        )

        new_population = []

        # Step 3: Neighbourhood update
        for p in range(population_size):

            # Find best solution in local neighbourhood
            left = (p - 1) % population_size
            right = (p + 1) % population_size

            neighbour_indices = [left, p, right]

            best_neighbour = max(
                neighbour_indices,
                key=lambda x: fitness_values[x]
            )

            local_best = population[best_neighbour]

            # Generate new configuration
            new_grid = create_new_configuration(
                population[p], local_best, k
            )

            # Selection
            old_fitness = fitness_values[p]

            new_fitness = calculate_fitness(
                new_grid, sunlight, penalty
            )

            if new_fitness > old_fitness:
                new_population.append(new_grid)
            else:
                new_population.append(population[p])

        population = new_population

        # Find best solution in current population
        fitness_values = parallel_fitness(
            population, sunlight, penalty
        )

        current_best_index = fitness_values.index(
            max(fitness_values)
        )

        current_best_fitness = fitness_values[
            current_best_index
        ]

        if current_best_fitness > global_best_fitness:

            global_best_fitness = current_best_fitness

            global_best = [
                row[:] for row in population[current_best_index]
            ]

        print(
            "Iteration", iteration + 1,
            "- Best Fitness:", global_best_fitness
        )

    return global_best, global_best_fitness


# ---------------------------------------
# 6. MAIN PROGRAM
# ---------------------------------------

print("SOLAR PANEL PLACEMENT OPTIMIZATION")

rows = int(input("Enter rooftop grid rows: "))
cols = int(input("Enter rooftop grid columns: "))

k = int(input("Enter maximum number of solar panels: "))

penalty = float(
    input("Enter shadow penalty per adjacent panel: ")
)

population_size = int(
    input("Enter population grid size: ")
)

iterations = int(
    input("Enter number of iterations: ")
)

# Read sunlight exposure values
sunlight = []

print("\nEnter sunlight values row by row:")

for i in range(rows):

    values = list(map(
        float,
        input("Row " + str(i + 1) + ": ").split()
    ))

    if len(values) != cols:
        raise ValueError("Enter exactly " + str(cols) +
                         " sunlight values per row.")

    sunlight.append(values)

if rows <= 0 or cols <= 0:
    raise ValueError("Rows and columns must be positive.")

if not (1 <= k <= rows * cols):
    raise ValueError("Invalid maximum number of panels.")

if population_size < 1 or iterations < 1:
    raise ValueError("Population size and iterations must be positive.")

# Run optimization
best_grid, best_fitness = solar_panel_optimization(
    sunlight,
    k,
    penalty,
    population_size,
    iterations
)

# ---------------------------------------
# 7. DISPLAY OUTPUT
# ---------------------------------------

print("\nOPTIMAL SOLAR PANEL PLACEMENT GRID")

for row in best_grid:
    print(row)

total_panels = sum(
    sum(row) for row in best_grid
)

print("\nTotal panels placed:", total_panels, "/", k)

energy = 0

for i in range(rows):
    for j in range(cols):
        if best_grid[i][j] == 1:
            energy += sunlight[i][j]

print("Total sunlight energy:", energy)

print("Maximum fitness:", best_fitness)


SOLAR PANEL PLACEMENT OPTIMIZATION
Enter rooftop grid rows: 4
Enter rooftop grid columns: 4
Enter maximum number of solar panels: 6
Enter shadow penalty per adjacent panel: 25
Enter population grid size: 3
Enter number of iterations: 3

Enter sunlight values row by row:
Row 1: 90 85 80 75
Row 2: 85 0 0 80
Row 3: 70 60 85 95
Row 4: 65 75 90 100
Iteration 1 - Best Fitness: 365.0
Iteration 2 - Best Fitness: 365.0
Iteration 3 - Best Fitness: 365.0

OPTIMAL SOLAR PANEL PLACEMENT GRID
[0, 0, 0, 0]
[1, 0, 0, 1]
[0, 0, 1, 0]
[1, 0, 0, 1]

Total panels placed: 5 / 6
Total sunlight energy: 415.0
Maximum fitness: 365.0
